# Week 4 — Understand ReAct by running an agent loop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week04/W4_lab_sql_manual.ipynb)

**Goal:** explain how an observation changes the next action, and how ReAct writes that connection as a **Thought**.

Start with the messages and one tool request you know from Week 3. First execute one exchange yourself; then repeat that same code in a loop; finally add ReAct to the same question. The comparison helps us identify the role of Thought. Basic can also solve the task.

| Stage | What you do | Approx. time |
|---|---|---|
| §1 Prepare | Load the supplied shop and tools | 10 min |
| §2 One exchange | Request → execute → save | 20 min |
| §3 Repeat | Read and run the visible loop | 15 min |
| §4 Add ReAct | Inspect Thought on the same question | 15 min |
| §5 Apply and check | Average order value; explain one decision | 20 min |
| §6 Investigate together | Category decrease → product comparison | +20 min |

Sections 1–5 take about 80 minutes. Section 6 adds about 20 minutes of instructor-led practice using the same data and loop.

The shop is **synthetic teaching data**, January–March 2025, in USD. Model replies are generated when you run the notebook; the labeled worked examples below are written for explanation.


## 1. Prepare the question, data, and tools

Open this notebook in Colab and choose **File → Save a copy in Drive**. Run the installation and setup cells below. The setup cell automatically downloads `w4_support.py`; no separate file upload is needed. In local Jupyter, you can also keep the helper beside the notebook using `W4_lab_bundle.zip`.

The provided module creates the synthetic database, evaluates arithmetic, describes the request format, and prints saved exchanges. It contains **no model calls or agent loop**. You do not need to study its implementation today.

Enter your API key in the **hidden input** when prompted, or use an existing `OPENAI_API_KEY` environment variable. The key is not written into notebook source or output. Each model request uses the OpenAI SDK; Python executes the returned tool request in the notebook.

In [ ]:
%pip install -q "openai==1.109.1"

In [ ]:
import json
import os
import sqlite3
from getpass import getpass
from pathlib import Path
from urllib.request import urlopen

# Download the supplied data/tool utilities when opening in a fresh Colab session.
SUPPORT_URL = "https://raw.githubusercontent.com/ralbu85/stml_2026/main/lectures/week04/w4_support.py"
if not Path("w4_support.py").is_file():
    with urlopen(SUPPORT_URL, timeout=30) as response:
        Path("w4_support.py").write_bytes(response.read())

from openai import OpenAI
from w4_support import prepare_shop, calculate, finish, action_tools, show_exchange, show_trace, DB_CONTEXT

API_KEY = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key (input hidden): ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4.1"
MAX_CALLS = 10
DB_PATH = prepare_shop(Path("outputs/w4_sql_manual/shop.sqlite"))
print("Ready:", DB_PATH)

### What is in the shop?

Our question is: **What were total sales from completed orders in March 2025?**

`orders` has one row per order, including its date and status. `order_items` has one row per product within an order, including quantity and unit price. Match them using `order_id`.

| orders: order_id | ordered_at | status |
|---|---|---|
| 182 | 2025-03-02 | completed |
| 183 | 2025-03-03 | completed |

| order_items: order_id | quantity | unit_price |
|---|---|---|
| 182 | 2 | 25 |
| 182 | 3 | 10 |
| 183 | 2 | 10 |
| 183 | 3 | 10 |

These are four actual rows from the generated data. Order 182 contributes `2 × 25 + 3 × 10 = 80` USD. For total sales, select **all completed March orders** and sum their item amounts. The sample alone is not the answer.

### What can Python execute?

| Operation | Input from the model | Result returned to the model |
|---|---|---|
| `run_sql` | One read-only SQL query | Column names, rows, and row count; or an error |
| `calculate` | Arithmetic such as `3055 / 57` | A number; or an error |
| `finish` | Final answer text | Submitted answer; an empty answer returns an error |

The next cell defines `run_sql`. It reads our database without changing it. A SQL error is returned as an observation that the model can use to revise its request.


In [ ]:
def run_sql(query: str) -> dict:
    """Execute one read-only SQLite query. Return all rows and row_count, or an error if the result exceeds 1,000 rows.

    Args:
        query: One read-only SQLite query using the supplied schema.
    """
    try:
        with sqlite3.connect(f'file:{DB_PATH.resolve()}?mode=ro', uri=True) as db:
            db.execute('PRAGMA query_only = ON')
            cursor = db.execute(query)
            rows = cursor.fetchmany(1001)
            if len(rows) > 1000:
                return {'error': 'Result exceeds 1,000 rows. No partial result was returned. Submit a query with a smaller result.'}
            return {'columns': [c[0] for c in cursor.description or []],
                    'rows': rows, 'row_count': len(rows)}
    except sqlite3.Error as error:
        return {'error': str(error)}


### Give the model the same information

`BASE` states the task rules; `DB_CONTEXT` lists the table columns and row meanings. `BASIC_TOOLS` describes the JSON request format for a native tool named `act`: **`action` chooses an operation; `action_input` holds its input**.

`operations` maps those names to Python functions. The schema describes a request; this dictionary lets our code execute it. Neither one selects the next action.


In [ ]:
BASE = """Answer the user's question using this shop database. Respond in English.
Sales are quantity * unit_price for completed orders; exclude cancelled orders.
Use SQL to filter, join, and aggregate database rows. Count the requested unit:
an order may have several item rows. Use calculate for arithmetic on returned
numbers when needed; SQL may also compute the final value directly.
Consult the data before answering. Reuse returned evidence; correct failed queries.
Submit the supported answer with finish, stating any unresolved limitation.
A successful query or finish does not prove that an answer is correct.
""" + DB_CONTEXT
BASIC_TOOLS = action_tools()
operations = {"run_sql": run_sql, "calculate": calculate, "finish": finish}
question = "What were total sales from completed orders in March 2025?"


## 2. Execute one exchange yourself

### A. Ask the model for a request

`messages` is the conversation sent to the model. It starts with the shared instructions (`system`) and our question (`user`).

The API returns a message containing one `act` tool call. `tool_choice="required"` requires a tool request; `parallel_tool_calls=False` limits it to one per call. These options **do not run Python**. Parse the JSON arguments into `request` and read what it asks for.

Before running: what month and status should a correct first SQL query select?


In [ ]:
messages = [
    {"role": "system", "content": BASE},
    {"role": "user", "content": question},
]
response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=BASIC_TOOLS,
    tool_choice="required", parallel_tool_calls=False,
    temperature=0, max_tokens=2400,
)
message = response.choices[0].message
if len(message.tool_calls or []) != 1:
    raise RuntimeError("Expected one tool request; inspect response before continuing.")
call = message.tool_calls[0]
request = json.loads(call.function.arguments)
print(json.dumps(request, indent=2))


### B. Execute that request in Python

For example, if `request["action"]` is `run_sql`, the dictionary selects our SQL function. The next line passes `action_input` to it and stores the returned value as `result`. This is the **Observation**.

No new model call occurs in this cell. Read the actual query and result together.


In [ ]:
result = operations[request["action"]](request["action_input"])
print(json.dumps(result, indent=2))


**Worked example — authored SQL, independently executed result.** Your model may write a different query.

```sql
SELECT SUM(quantity * unit_price) AS sales
FROM orders JOIN order_items USING (order_id)
WHERE status = 'completed' AND ordered_at LIKE '2025-03%';
```

It returns `{"columns": ["sales"], "rows": [[3055.0]], "row_count": 1}`. This means **one returned total of 3,055 USD**. `row_count=1` does not mean one order. If your result differs, inspect the filters and measured quantity. If it is an error, the next model request should repair the query.

### C. Save the request and result for the next call

The model does not automatically see what Python returned. Append **both** messages: the assistant's tool request and the tool's result. `tool_call_id` connects a result to its request; `model_dump` converts the SDK message into a dictionary without dropping its arguments.


In [ ]:
messages.append(message.model_dump(exclude_none=True))
messages.append({"role": "tool", "tool_call_id": call.id,
                 "content": json.dumps(result)})
print("Saved roles:", [m["role"] for m in messages])
show_trace(messages)


You should now see `system → user → assistant → tool`. The next API call can use this updated `messages` list to decide whether to query again or submit an answer.

**Pause:** which line executes Python? Which two lines make the next model call aware of that execution? What would the model be missing if you saved only the request?

Run A → B → C in order once. To retry this demonstration, restart at A; rerunning only C duplicates the exchange. If the API fails in A, fix the key, network, or access error before continuing.

## 3. Repeat the same exchange in a loop

`run_loop` below places A → B → C inside `for`. It starts a **fresh conversation** for the supplied question, so the manual demonstration stays separate. Each next iteration receives the list updated by the previous iteration.

The return value contains `answer`, saved `messages`, and the number of model `calls`. A successful `finish` returns the submitted answer. Errors stay in the history so the next request can address them. At the call limit, the function returns `answer=None`: stopping is not the same as answering.

Read the code and locate the four operations: **model request → Python execution → history update → stopping decision**.


In [ ]:
def run_loop(question, instructions=BASE, tools=BASIC_TOOLS, max_calls=MAX_CALLS):
    messages = [{"role": "system", "content": instructions},
                {"role": "user", "content": question}]
    for turn in range(max_calls):
        # A. Ask for the next request using all messages so far.
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice="required", parallel_tool_calls=False,
            temperature=0, max_tokens=2400,
        )
        message = response.choices[0].message
        if len(message.tool_calls or []) != 1:
            raise RuntimeError("Expected one tool request; inspect response.")
        call = message.tool_calls[0]
        request = json.loads(call.function.arguments)
        # B. Execute the selected Python operation.
        result = operations[request["action"]](request["action_input"])
        # C. Save both sides of the exchange for the next model call.
        messages.append(message.model_dump(exclude_none=True))
        messages.append({"role": "tool", "tool_call_id": call.id,
                         "content": json.dumps(result)})
        print(f"\n--- Exchange {turn + 1} ---")
        show_exchange(request, result)
        # D. Stop only after a successful answer submission.
        if request["action"] == "finish" and "answer" in result:
            return {"answer": result["answer"], "messages": messages, "calls": turn + 1}
    print("Call limit reached without a submitted answer.")
    return {"answer": None, "messages": messages, "calls": max_calls}


Run the total-sales question from the beginning. Read the first SQL observation and predict the next request before reading it. The model may retrieve more information; it should finish once the question is supported.


In [ ]:
basic = run_loop(question)
print("Basic answer:", basic["answer"])


## 4. Add ReAct to the same loop

A **Thought** is the model's written assessment of the evidence, remaining uncertainty, and next action. In ReAct, this assessment is updated after an Observation and guides the next Action or `finish`. Written reasoning can be wrong; check it against the query and returned values.

We change only the reasoning instruction and request format. `REACT_TOOLS` adds a `thought` field before `action` and `action_input`. The loop, question, model, data, operations, temperature, and limits remain the same. The loop saves the full request, so the Thought is also part of the next call's history.


In [ ]:
REACT = """
Before each action, write thought: assess the evidence available so far,
what is still missing, and why this next action addresses it.
After an observation, check what the query or calculation actually measured.
If it was wrong, revise the next action. Before finish, explain how the
observations support each requested part of the answer or leave a limitation.
"""
REACT_TOOLS = action_tools(react=True)
react = run_loop(question, instructions=BASE + REACT, tools=REACT_TOOLS)
print("ReAct answer:", react["answer"])


### Find where the Observation changed the next decision

The following cell **only reads saved messages**; it makes no API or tool calls. Choose one SQL result and read the next request. Basic may choose the same action. Identify what the added Thought explains.

For example, after the total-sales query returns 3055, an authored Thought could be: *“The query selected completed March orders and summed their item amounts. The returned total answers the question, so I can finish.”* This is an explanation to assess, not proof that ReAct is more accurate.


In [ ]:
show_trace(react["messages"])
# To inspect the Basic exchange instead: show_trace(basic["messages"])


## 5. Apply it: average value per order

Change the question: **What was the average value of a completed order in March 2025?**

Keep the same month and order status. We already know the numerator is total sales. What should the denominator count? Order 182 has two item rows but is still **one order**. Predict the required SQL result before running.

The two calls below each start a fresh conversation. A model may request total sales and order count, then calculate their ratio; it may also compute the correct average directly in SQL.


In [ ]:
average_question = "What was the average value of a completed order in March 2025?"
basic_average = run_loop(average_question)
react_average = run_loop(average_question, instructions=BASE + REACT, tools=REACT_TOOLS)


### Check the answer independently

The next cell contains an instructor-written reference query. It calls only the local SQL and calculator functions, **not the model**. It does not enter either agent's history or fix its answer.

After joining items, `COUNT(*)` counts item rows; `COUNT(DISTINCT order_id)` counts each order once. Inspect the filters as well as the numbers.


In [ ]:
reference = run_sql("""
SELECT COUNT(*) AS item_rows, COUNT(DISTINCT order_id) AS orders,
       SUM(quantity * unit_price) AS sales
FROM orders JOIN order_items USING (order_id)
WHERE status = 'completed' AND ordered_at LIKE '2025-03%';
""")
print(reference)
print(calculate("3055 / 57"))


The reference is **116 item rows, 57 orders, and 3,055 USD**; the average is **53.60 USD per completed order**, rounded. A valid submitted answer still needs to match the requested unit and population.

**Use one saved exchange to answer:**

1. What did its query actually measure? Which result supports that interpretation?
2. What did the next Thought infer, and did the next action follow from it?
3. If both methods succeeded, what does the Thought make explicit? One pair does not establish an accuracy advantage.
4. If a query failed or the answer was wrong, did the later request repair the error? If `answer` is `None`, inspect the last exchange and explain why the budget ended.

**Try one change:** replace March with February in `average_question` and rerun the two calls in the previous cell. Change `2025-03%` to `2025-02%` in the independent query too. Compute sales divided by distinct orders using the newly returned numbers; do not reuse March's answer. Explain one transition before comparing call counts.

## 6. Investigate together: where did sales decrease?

We have measured total sales and average order value. Now compare **February and March 2025** and use the first result to choose what to investigate next.

The question is where the decrease appears in the sales records. These records do not contain inventory levels, advertising, or customer reasons.

Use the same `run_loop`, ReAct instructions, tools, and database. We already compared Basic and ReAct; these two activities concentrate on interpreting a saved Observation and the next Thought or action.

`products` supplies **`category`** and **`name`**. Join it to `order_items` by `product_id`, and join the items to `orders` by `order_id` for dates and completion status. Its categories are Electronics, Home, and Stationery.

### 6.1 Compare categories before selecting a product

**Before running together:** the total can reveal that sales decreased, but can it reveal which category changed? What should the next query return?

Request both months and define **change = March sales − February sales**. A negative change is a decrease.

In [ ]:
category_question = (
    "Compare completed sales in February and March 2025 by category. "
    "Show each month's sales and the change (March minus February), "
    "plus the overall change. Which category contributed most to the decrease?"
)
category_run = run_loop(category_question, instructions=BASE + REACT, tools=REACT_TOOLS)
print("Category comparison:", category_run["answer"])

Read one category query and its returned values, then read the next Thought.

- Did the query select both months and completed orders?
- Is there enough evidence to answer the category question, or is another operation needed?
- Which category should we investigate next, and which returned number supports that choice?

A model may calculate all requested values in one SQL query. That is valid if the result supports the answer; extra tool calls are not a goal. If `answer` is `None`, use the saved trace to find what remains unresolved.

The next cell redisplays saved messages without another API call.

In [ ]:
show_trace(category_run["messages"])

### Check the category values independently

This instructor-written query runs locally. It does not enter the model's history or change its answer. `CASE` places each item's amount in the corresponding month's sum.

In [ ]:
category_reference = run_sql("""
WITH category_sales AS (
    SELECT p.category,
           SUM(CASE WHEN o.ordered_at LIKE '2025-02%'
                    THEN i.quantity * i.unit_price ELSE 0 END) AS feb_sales,
           SUM(CASE WHEN o.ordered_at LIKE '2025-03%'
                    THEN i.quantity * i.unit_price ELSE 0 END) AS mar_sales
    FROM orders AS o
    JOIN order_items AS i USING (order_id)
    JOIN products AS p USING (product_id)
    WHERE o.status = 'completed'
      AND o.ordered_at >= '2025-02-01' AND o.ordered_at < '2025-04-01'
    GROUP BY p.category
)
SELECT category, feb_sales, mar_sales, mar_sales - feb_sales AS change
FROM category_sales
ORDER BY change;
""")
print(json.dumps(category_reference, indent=2))

For the supplied data, the independent result is:

| Category | February sales (USD) | March sales (USD) | Change (USD) |
|---|---:|---:|---:|
| Electronics | 5,710 | 280 | −5,430 |
| Stationery | 1,075 | 835 | −240 |
| Home | 1,345 | 1,940 | +595 |
| **Total** | **8,130** | **3,055** | **−5,075** |

**Discuss together:** why can the Electronics decrease exceed the overall decrease? Home increased, offsetting part of the decline. Check that `−5,430 − 240 + 595 = −5,075`.

Electronics is the category with the largest decrease. We now have a reason to inspect its products; the total alone did not identify that next target.

### 6.2 Use that finding to choose the next question

`run_loop` starts a **fresh conversation** each time. The previous call's messages are not automatically included. Put the category selected from the observed result into `CATEGORY_TO_INSPECT`; the new question also states the dates explicitly.

For the main walkthrough use Electronics. Home and Stationery are available for an optional follow-up. The category question is now answered; the class is asking a more detailed product question.

The prompt includes an SQL hint: total the eligible sales first, then join those totals to the product list. This keeps the focus on reading the Observation and judging the next action.

**Before running:** if a product sold in February but not in March, should it disappear from the comparison, or appear with March sales of zero?

In [ ]:
CATEGORY_TO_INSPECT = "Electronics"  # Choose from the category result above.
assert CATEGORY_TO_INSPECT in {"Electronics", "Home", "Stationery"}
product_question = (
    f"For {CATEGORY_TO_INSPECT}, compare completed sales by product in February "
    "and March 2025. Show each month's sales and the change (March minus February). "
    "Keep every product in this category, showing zero for a month with no "
    "completed sales. Aggregate completed order items by product and month first, "
    "then join those totals to the full product list and fill missing totals with zero. "
    "Which products sold in February but not March? "
    "What can these sales records establish, and what remains unknown "
    "about why the sales changed?"
)
product_run = run_loop(product_question, instructions=BASE + REACT, tools=REACT_TOOLS)
print("Product comparison:", product_run["answer"])

Inspect the SQL, its Observation, and the following Thought.

- Does the result include products with zero completed sales in March? Do the product totals agree with the category result above?
- Does the Thought interpret zero as **no completed sales in the period**, or invent a reason such as a stockout?
- Does the next action obtain needed evidence, or finish with a supported answer and a clear limit?

If the query failed, inspect whether the next request uses the error to repair it. A query that runs successfully can still omit the products we needed to compare.

In [ ]:
show_trace(product_run["messages"])

### Check that products with zero sales stay in the result

The reference below first totals completed sales across both months. It then starts from **all products in the selected category** and uses `LEFT JOIN` to keep products even when the totals are absent. `COALESCE(..., 0)` displays a missing total as zero.

This is a reference query, not a required query template for the model. Other SQL is valid if it returns the same requested products and values. It uses the same `CATEGORY_TO_INSPECT` selected above.

**A possible SQL error without an exception:** restricting `orders` in a `LEFT JOIN` does not remove unmatched `order_items`. Summing those item amounts can include the wrong dates or cancelled orders. Filtering and aggregating the eligible sales first avoids this mistake.

In [ ]:
product_reference = run_sql(f"""
WITH product_sales AS (
    SELECT i.product_id,
           SUM(CASE WHEN o.ordered_at LIKE '2025-02%'
                    THEN i.quantity * i.unit_price ELSE 0 END) AS feb_sales,
           SUM(CASE WHEN o.ordered_at LIKE '2025-03%'
                    THEN i.quantity * i.unit_price ELSE 0 END) AS mar_sales
    FROM orders AS o JOIN order_items AS i USING (order_id)
    WHERE o.status = 'completed'
      AND o.ordered_at >= '2025-02-01' AND o.ordered_at < '2025-04-01'
    GROUP BY i.product_id
)
SELECT p.name,
       COALESCE(s.feb_sales, 0) AS feb_sales,
       COALESCE(s.mar_sales, 0) AS mar_sales,
       COALESCE(s.mar_sales, 0) - COALESCE(s.feb_sales, 0) AS change
FROM products AS p
LEFT JOIN product_sales AS s USING (product_id)
WHERE p.category = '{CATEGORY_TO_INSPECT}'
ORDER BY change;
""")
print(json.dumps(product_reference, indent=2))

For **Electronics**, compare your result with these independently checked values:

| Product | February sales (USD) | March sales (USD) | Change (USD) |
|---|---:|---:|---:|
| Headphones | 2,400 | 0 | −2,400 |
| Keyboard | 2,300 | 0 | −2,300 |
| Mouse | 650 | 0 | −650 |
| Cable | 360 | 280 | −80 |

Their changes total **−5,430 USD**, matching the category result. Headphones, Keyboard, and Mouse account for −5,350 USD; Cable contributes the remaining −80 USD. A March-only list would miss the first three products.

These records establish the change in **completed sales**, not whether the products were out of stock, removed from sale, or no longer wanted. Those claims would require additional data. It is appropriate to finish the answer with that limitation.

**Close the activity together:** point to one Observation that justified a next action or a final answer. Then explain why the class selected Electronics for the second question. The two questions are connected by that finding; they do not share a conversation automatically.

If time permits, select Home in `CATEGORY_TO_INSPECT` and rerun the product activity and reference. Explain the increases using the newly returned values.

### Sources

API message pattern: [OpenAI function-calling guide](https://developers.openai.com/api/docs/guides/function-calling). The visible loop is course-authored. ReAct concept: [Yao et al., ReAct](https://arxiv.org/abs/2210.03629).